In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install gensim wandb wikipedia-api langchain langchain_text_splitters langchain-community langchain-huggingface faiss-cpu transformers accelerate bitsandbytes --quiet
!pip install --upgrade transformers

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
import wandb
 
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
 
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nMissing values in train:\n", train_df.isnull().sum())
print("\nAnswer label distribution:\n", train_df["answer"].value_counts())
 
train_df["prompt_len"] = train_df["prompt"].astype(str).apply(lambda x: len(x.split()))
print("\nPrompt word-length stats:\n", train_df["prompt_len"].describe())

print(train_df.head(3))

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
 
 
TEXT_COLS = ["prompt", "A", "B", "C", "D", "E"]
 
for col in TEXT_COLS:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

def tokenize(text):
    return text.split()

In [ ]:
all_sentences = []
for df in [train_df, test_df]:
    for col in TEXT_COLS:
        all_sentences.extend(df[col].apply(tokenize).tolist())
 
EMBED_DIM = 100
 
w2v_model = Word2Vec(
    sentences=all_sentences,
    vector_size=EMBED_DIM,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=SEED,
)
 
print("Vocabulary size:", len(w2v_model.wv))
w2v_model.save(os.path.join("/kaggle/working", "word2vec.model"))

In [ ]:
def text_to_vector(text, model, dim=EMBED_DIM):
    words = tokenize(text)
    vecs = []

    for word in words:
        if word in model.wv:
            vecs.append(model.wv[word])

    if len(vecs) == 0:
        return np.zeros(dim, dtype=np.float32)

    avg_vector = np.mean(vecs, axis=0)
    return avg_vector.astype(np.float32)

In [ ]:
LABELS = ["A", "B", "C", "D", "E"]
LABEL2IDX = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

class MCQDataset(Dataset):

    def __init__(self, df, w2v_model, has_labels=True):
        self.df = df.reset_index(drop=True)
        self.model = w2v_model
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        question_vector = text_to_vector(row["prompt"], self.model)

        features = []

        for option in LABELS:
            option_vector = text_to_vector(row[option], self.model)

            difference = np.abs(question_vector - option_vector)

            feature = np.concatenate((question_vector, option_vector, difference))

            features.append(feature)

        features = np.array(features)

        data = {}
        data["features"] = torch.tensor(features, dtype=torch.float32)

        if self.has_labels:
            answer = LABEL2IDX[row["answer"]]
            data["label"] = torch.tensor(answer, dtype=torch.long)
        else:
            data["id"] = row["id"]

        return data

In [ ]:
class MCQScorer(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
 
    def forward(self, x):
        batch, n_options, dim = x.shape
        x = x.view(batch * n_options, dim)
        scores = self.net(x)
        scores = scores.view(batch, n_options)
        return scores

In [ ]:
def map_at_3(probs, labels):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = []

    for pred, true in zip(top3, labels):
        if true in pred:
            score.append(1 / (np.where(pred == true)[0][0] + 1))
        else:
            score.append(0)

    return np.mean(score)


def run_epoch(model, loader, optimizer, criterion, train=True):

    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0
    all_probs = []
    all_labels = []

    for batch in loader:

        x = batch["features"].to(DEVICE)
        y = batch["label"].to(DEVICE)

        with torch.set_grad_enabled(train):

            output = model(x)
            loss = criterion(output, y)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * x.size(0)

        all_probs.append(torch.softmax(output, dim=1).cpu().detach().numpy())
        all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    loss = total_loss / len(loader.dataset)
    acc = (all_probs.argmax(1) == all_labels).mean()
    map3 = map_at_3(all_probs, all_labels)

    return loss, acc, map3


try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except:
    pass

wandb.login()

wandb.init(
    project="dlgenai-project-26t2",
    config={
        "batch_size": 32,
        "epochs": 20,
        "lr": 1e-3,
        "hidden_dim": 128
    }
)

cfg = wandb.config

train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,
    random_state=SEED,
    stratify=train_df["answer"]
)

train_loader = DataLoader(
    MCQDataset(train_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=True
)

val_loader = DataLoader(
    MCQDataset(val_data, w2v_model),
    batch_size=cfg.batch_size,
    shuffle=False
)

model = MCQScorer(3 * EMBED_DIM, cfg.hidden_dim).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
criterion = nn.CrossEntropyLoss()

best_map = 0

for epoch in range(cfg.epochs):

    train_loss, train_acc, train_map = run_epoch(
        model, train_loader, optimizer, criterion, True
    )

    val_loss, val_acc, val_map = run_epoch(
        model, val_loader, optimizer, criterion, False
    )

    wandb.log({
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_map3": train_map,
        "val_map3": val_map
    })

    print(f"Epoch {epoch+1}  Validation MAP@3 = {val_map:.4f}")

    if val_map > best_map:
        best_map = val_map
        torch.save(model.state_dict(), "/kaggle/working" + "/best_model.pt")

wandb.finish()

In [ ]:
model = MCQScorer(3 * EMBED_DIM, 128).to(DEVICE)
model.load_state_dict(torch.load("/kaggle/working" + "/best_model.pt", map_location=DEVICE))
model.eval()

test_loader = DataLoader(
    MCQDataset(test_df, w2v_model, has_labels=False),
    batch_size=32,
    shuffle=False
)

ids = []
predictions = []

with torch.no_grad():

    for batch in test_loader:

        x = batch["features"].to(DEVICE)

        probs = torch.softmax(model(x), dim=1).cpu().numpy()
        top3 = np.argsort(-probs, axis=1)[:, :3]

        for i in range(len(top3)):
            ids.append(int(batch["id"][i]))
            predictions.append(" ".join(LABELS[j] for j in top3[i]))

test_df["Prediction_Model_NN"] = predictions
print("NN Model predictions saved...")

In [ ]:
wiki_topics = [
    "Supersymmetric quantum mechanics", "Heisenberg uncertainty principle", "Virtual particle",
    "Spontaneous symmetry breaking", "Wigner distribution function", "Magnetic monopole",
    "Spin quantum number", "Parity (physics)", "Peierls bracket", "Geometric quantization",
    "Quantum field theory", "Hilbert space", "Probability amplitude", "Ramsauer–Townsend effect",
    "Explicit symmetry breaking", "Angular momentum operator", "Standard Model",
    "Higgs boson", "CP violation", "Quark", "Chemical potential",
    "Lorentz covariance", "Minkowski space", "Minkowski diagram", "Special relativity",
    "General relativity", "Simultaneity", "Speed of light", "Born reciprocity",
    "Frame-dragging", "Gravitomagnetism", "Gravity Probe B", "Roche limit",
    "Penrose process", "Black hole information paradox", "Schwarzschild black hole",
    "CEERS-93316", "James Webb Space Telescope", "Redshift", "Metric expansion of space",
    "Proper distance", "Interstellar medium", "Molecular cloud", "Supernova remnant",
    "Main sequence", "Pulsar", "Crab Pulsar", "Supermassive black hole",
    "Sagittarius A*", "Dark matter", "Gravitational wave", "Doppler effect",
    "Lyman-alpha line", "Planetary system", "X-ray pulsar-based navigation",
    "Baryon acoustic oscillations", "Modified Newtonian dynamics", "Inflaton",
    "Einstein@Home", "Light-year", "Apparent magnitude", "Metallicity",
    "Kapteyn's Star", "Isophote", "Type Ia supernova", "Supernova",
    "Carnot heat engine", "Maxwell's demon", "Throttling process", "Second law of thermodynamics",
    "Kelvin–Helmholtz instability", "Coherent turbulent structure", "Cavitation", "Convection",
    "Natural convection", "Bernoulli's principle", "Kutta condition", "Navier–Stokes equations",
    "Cauchy momentum equation", "Water hammer",
    "Fermat's principle", "Emissivity", "Illuminance", "Luminance", "Rayleigh scattering",
    "Young's interference experiment", "Diffraction", "Total internal reflection",
    "Radiosity (radiometry)", "Stefan–Boltzmann law", "Ultraviolet catastrophe",
    "Optical signal-to-noise ratio", "Propagation constant", "Loudness",
    "Landau–Lifshitz–Gilbert equation", "Magnetic susceptibility", "Memristor",
    "Spin valve", "Electrical resistivity and conductivity", "Superconductivity",
    "Amorphous metal", "Variable-range hopping", "Piezoelectricity", "Dielectric loss",
    "Josephson effect", "De Haas–Van Alphen effect", "Paramagnetism", "Order parameter",
    "Ferroelectricity", "ReRAM", "Evans balance", "Spatial dispersion",
    "Identity element", "Crystallographic point group", "Improper rotation",
    "Crystallinity", "API gravity", "Radiometric dating", "Recrystallization (metallurgy)", "Grain boundary strengthening", 
    "Fischer–Tropsch process", "Carbocation", "Naphthalene", "Crossover experiment",
    "Fourier-transform infrared spectroscopy", "Three moment theorem", "Bollard pull", "Ring-imaging Cherenkov detector", 
    "Formal system", "Uniform tilings in hyperbolic plane", "Regular polytope",
    "Probability density function", "Probability mass function", "Reciprocal length",
    "Symmetry group", "Erlangen program", "Hyperbolic geometry", "Permutation group",
    "CW complex", "Dimension", "Hesse's principle of transfer", "Dynamic scaling", "Liouville's theorem (Hamiltonian)", 
    "Surgical pathology", "Active transport", "Trophic level", "Pulmonary circulation",
    "Mammary gland", "Organography", "Cyclotide", "Phageome",
    "Myrmecophyte", "Mycorrhiza", "IL-10", "Regulatory T cell", "Anatomy", "Cardiac skeleton",
    "Second", "Coordinated Universal Time", "Universal Time",
    "Triskelion", "Newton's laws of motion", "Right-hand rule", "Giordano Bruno",
    "Shower-curtain effect", "Wilson cloud chamber", "Ozma Problem", "Horror vacui",
    "Butterfly effect", "Gauss's law", "Scale (map)", "Martin Heidegger",
    "Isaac Newton", "Robert Hooke", "Pierre de Fermat",
    "Classical mechanics", "Earnshaw's theorem", "Environmental Science Center",
    "Memristor", "Synaptic transistor", "Power density", "Cold dark matter", "Antimatter",
    "Baryon asymmetry", "L dwarf",
    "Pycnometer", "Photophoresis", "Isophote", "Recycling", "Rare-earth element",
    "Fusor", "Thylakoid", "Diquark", "Thermodynamic system", "Molecular symmetry",
    "Mass-to-charge ratio", "Rømer's determination of the speed of light",
    "Resistive random-access memory", "Diffuse sky radiation", "Grain growth",
]

In [ ]:
import os
import wikipediaapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

wiki = wikipediaapi.Wikipedia(
    user_agent="MyRAGProject/1.0 (singhshikhar8957@gmail.com)",
    language="en"
)

print("Scraping Wikipedia to build the Knowledge Base...")

scraped_texts = []

for topic in wiki_topics:
    try:
        page = wiki.page(topic)
        if page.exists():
            scraped_texts.append(page.text)
        else:
            print(f"Skipped {topic}: page not found")
    except Exception as e:
        print(f"Skipped {topic} due to error: {e}")
print(f"Successfully scraped {len(scraped_texts)} articles.")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=300)
docs = text_splitter.create_documents(scraped_texts)
print(f"Created {len(docs)} chunks.")

embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", 
                                   model_kwargs={"device": DEVICE,
                                                "model_kwargs": {"use_safetensors": False}},
                                   encode_kwargs={"normalize_embeddings": True}
)
vector_db = FAISS.from_documents(docs, embeddings)

FAISS_SAVE_PATH = "/kaggle/working/faiss_index"
vector_db.save_local(FAISS_SAVE_PATH)
print(f"FAISS index created successfully and saved to {FAISS_SAVE_PATH}")

In [ ]:
import transformers
from transformers import AutoTokenizer, AutoModelForMultimodalLM, pipeline, logging
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
import torch

transformers.logging.set_verbosity_error()

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(hf_token)
    print("Logged into Hugging Face successfully!")
except Exception as e:
    print(f"HF login failed: {e}. Please ensure you have added a 'HF_TOKEN' secret in Kaggle.")

model_id = "google/gemma-4-12B-it-qat-q4_0-unquantized" 

print(f"Loading {model_id} from Hugging Face...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

llm_model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

llm_pipe = pipeline(
    "text-generation", 
    model=llm_model, 
    tokenizer=tokenizer, 
    max_new_tokens=20, 
    do_sample=False, 
    return_full_text=False, 
    pad_token_id=tokenizer.eos_token_id
)
print("LLM loaded successfully and ready for inference!")

In [ ]:
print("Loading offline FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5", 
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)

FAISS_SAVE_PATH = "/kaggle/working/faiss_index"

loaded_vector_db = FAISS.load_local(
    FAISS_SAVE_PATH, 
    embeddings, 
    allow_dangerous_deserialization=True
)
print("Knowledge Base loaded successfully!")

In [ ]:
import pandas as pd
from langchain_core.documents import Document

print("Building Train Dataset Vector DB...")
train_docs = []

for _, row in train_df.iterrows():
    train_text = (
        f"Question: {row['prompt']}\n"
        f"A: {row['A']}\nB: {row['B']}\nC: {row['C']}\nD: {row['D']}\nE: {row['E']}\n"
        f"Correct Answer: {row['answer']}"
    )
    doc = Document(page_content=train_text)
    train_docs.append(doc)

train_vector_db = FAISS.from_documents(train_docs, embeddings)
print("Train Dataset Vector DB created successfully!")

In [ ]:
def retrieve_context(question, k=3):
    docs = loaded_vector_db.similarity_search(question, k=k)
    return "\n".join(d.page_content for d in docs)

def retrieve_train_examples(question, k=2):
    docs = train_vector_db.similarity_search(question, k=k)
    return "\n\n".join(d.page_content for d in docs)

def build_prompt(question, options, wiki_context, train_examples):
    options_text = ""
    for label in ["A", "B", "C", "D", "E"]:
        if label in options:
            options_text += label + ": " + options[label] + "\n"
            
    prompt = (
        "Wiki Context:\n" + wiki_context + "\n\n"
        "Similar Examples from Training Data (Pay close attention to the Correct Answer here):\n" + train_examples + "\n\n"
        "Question: " + question + "\n\n" + options_text + "\n"
        "Based on the similar examples and Wiki context above, rank the 3 most likely correct options.\n"
        "Reply with exactly 3 letters separated by spaces, nothing else.\n"
        "Answer:"
    )
    return prompt

def generate_answer(prompt):
    return llm_pipe(prompt, max_new_tokens=10)[0]['generated_text']

def parse_letters(raw_output, valid_labels):
    found = re.findall(r'\b[A-E]\b', raw_output.upper())
    letters = []
    for ch in found:
        if ch in valid_labels and ch not in letters:
            letters.append(ch)
    return letters

def normalize(text):
    text = text.lower()
    return re.sub(r"[^a-z0-9\s]", " ", text)

def token_overlap_score(option_text, context_text):
    opt_tokens = set(normalize(option_text).split())
    ctx_tokens = set(normalize(context_text).split())
    if not opt_tokens:
        return 0.0
    return len(opt_tokens & ctx_tokens) / len(opt_tokens)

def similarity_backup_ranking(options, context):
    scores = {}
    for label, text in options.items():
        scores[label] = token_overlap_score(text, context)
    return sorted(scores, key=scores.get, reverse=True)

def answer_question(row):
    question = row["prompt"]
    options = {}
    for k in ["A", "B", "C", "D", "E"]:
        if k in row and str(row[k]) != "nan":
            options[k] = str(row[k])

    wiki_context = retrieve_context(question)
    train_examples = retrieve_train_examples(question, k=2)
    prompt = build_prompt(question, options, wiki_context, train_examples)
    raw_output = generate_answer(prompt)
    letters = parse_letters(raw_output, list(options.keys()))
    backup = similarity_backup_ranking(options, wiki_context)
    for label in backup:
        if len(letters) >= 3:
            break
        if label not in letters:
            letters.append(label)

    return " ".join(letters[:3])

predictions = []
for _, row in test_df.iterrows():
    pred = answer_question(row)
    predictions.append(pred)

test_df["Final_Prediction"] = predictions

In [ ]:
final_submission = pd.DataFrame({
    "ID": test_df["id"],
    "Prediction": test_df["Final_Prediction"]})

final_submission.to_csv("submission.csv", index=False)
print(final_submission.head())